# Hayato Dashboard Spreadsheet Data

Google Drive の `MyDrive/ハヤトの野望/data/` に、GAS + DuckDB-Wasm 検証用の Google Sheets を作成する。


In [ ]:
!pip -q install gspread google-api-python-client pandas


In [ ]:
from google.colab import auth
import google.auth
import gspread
from googleapiclient.discovery import build

auth.authenticate_user()

SCOPES = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive',
]
creds, _ = google.auth.default(scopes=SCOPES)
gc = gspread.authorize(creds)
drive_service = build('drive', 'v3', credentials=creds)


In [ ]:
from datetime import datetime, timedelta, timezone
import pandas as pd

FOLDER_ID = '1BMLDFq1tZ7Jj69Pf0FMks80JTRTtC9S1'
SPREADSHEET_NAME = 'hayato_live_spreadsheet'
SPREADSHEET_FOLDER_PATH = 'data'
ACTIVE_SHEET_NAME = 'active'
STAGING_SHEET_NAME = 'staging'
METADATA_SHEET_NAME = 'metadata'
ROW_COUNT = 10_000

base_rows = [
    {
        'video_id': 'demo_live_001',
        'title': 'なんでもできる超リアルな都市で好き放題してみた',
        'published_at': '2026-04-30T12:00:00+09:00',
        'duration_sec': 11630,
        'live_type': 'LIVE',
        'visibility': 'public',
        'peak_concurrent_viewers': 2688,
        'view_count': 180000,
        'like_count': 3200,
        'comment_count': 240,
        'thumbnail_url': 'https://i.ytimg.com/vi/dQw4w9WgXcQ/mqdefault.jpg',
    },
    {
        'video_id': 'demo_live_002',
        'title': 'ハヤトの野望作戦会議',
        'published_at': '2026-04-28T12:00:00+09:00',
        'duration_sec': 7579,
        'live_type': 'LIVE',
        'visibility': 'public',
        'peak_concurrent_viewers': 1284,
        'view_count': 95000,
        'like_count': 1800,
        'comment_count': 130,
        'thumbnail_url': 'https://i.ytimg.com/vi/aqz-KE-bpKQ/mqdefault.jpg',
    },
    {
        'video_id': 'demo_live_003',
        'title': 'shapez2 無限立体構造',
        'published_at': '2026-04-24T12:00:00+09:00',
        'duration_sec': 12941,
        'live_type': 'LIVE',
        'visibility': 'public',
        'peak_concurrent_viewers': 2513,
        'view_count': 140000,
        'like_count': 2600,
        'comment_count': 190,
        'thumbnail_url': 'https://i.ytimg.com/vi/M7lc1UVf-VE/mqdefault.jpg',
    },
    {
        'video_id': 'demo_live_004',
        'title': '戦国武将になって天下統一を目指す #2',
        'published_at': '2026-04-17T12:00:00+09:00',
        'duration_sec': 11912,
        'live_type': 'LIVE',
        'visibility': 'public',
        'peak_concurrent_viewers': 2816,
        'view_count': 210000,
        'like_count': 4100,
        'comment_count': 320,
        'thumbnail_url': 'https://i.ytimg.com/vi/dQw4w9WgXcQ/mqdefault.jpg',
    },
    {
        'video_id': 'demo_live_005',
        'title': '戦国武将になって天下統一を目指す',
        'published_at': '2026-04-16T12:00:00+09:00',
        'duration_sec': 12344,
        'live_type': 'LIVE',
        'visibility': 'public',
        'peak_concurrent_viewers': 3365,
        'view_count': 240000,
        'like_count': 4800,
        'comment_count': 380,
        'thumbnail_url': 'https://i.ytimg.com/vi/aqz-KE-bpKQ/mqdefault.jpg',
    },
]

rows = []
base_time = datetime(2026, 4, 30, 12, 0, 0, tzinfo=timezone(timedelta(hours=9)))

for i in range(ROW_COUNT):
    base = base_rows[i % len(base_rows)].copy()
    base['video_id'] = f"{base['video_id']}_{i + 1:05d}"
    base['title'] = f"{base['title']} #{i + 1:05d}"
    base['published_at'] = (base_time - timedelta(days=i % 365)).isoformat()
    base['duration_sec'] = int(base['duration_sec'] + (i % 1800))
    base['peak_concurrent_viewers'] = int(base['peak_concurrent_viewers'] + ((i * 37) % 5000))
    base['view_count'] = int(base['view_count'] + ((i * 997) % 300000))
    base['like_count'] = int(base['like_count'] + ((i * 41) % 8000))
    base['comment_count'] = int(base['comment_count'] + ((i * 7) % 600))
    rows.append(base)

df = pd.DataFrame(rows)[[
    'video_id',
    'title',
    'published_at',
    'duration_sec',
    'live_type',
    'visibility',
    'view_count',
    'like_count',
    'comment_count',
    'peak_concurrent_viewers',
    'thumbnail_url',
]]

df.head()


In [ ]:
def find_folder_in_folder(parent_folder_id, folder_name):
    query = (
        f"'{parent_folder_id}' in parents and "
        f"name = '{folder_name}' and "
        "mimeType = 'application/vnd.google-apps.folder' and "
        "trashed = false"
    )
    response = drive_service.files().list(
        q=query,
        spaces='drive',
        fields='files(id, name)',
        pageSize=10,
    ).execute()
    folders = response.get('files', [])

    if len(folders) > 1:
        raise RuntimeError(f'Duplicate folder name: {folder_name}')

    return folders[0] if folders else None


def get_or_create_folder(parent_folder_id, folder_name):
    folder = find_folder_in_folder(parent_folder_id, folder_name)

    if folder:
        return folder

    return drive_service.files().create(
        body={
            'name': folder_name,
            'mimeType': 'application/vnd.google-apps.folder',
            'parents': [parent_folder_id],
        },
        fields='id, name',
    ).execute()


def find_spreadsheet_in_folder(folder_id, spreadsheet_name):
    query = (
        f"'{folder_id}' in parents and "
        f"name = '{spreadsheet_name}' and "
        "mimeType = 'application/vnd.google-apps.spreadsheet' and "
        "trashed = false"
    )
    response = drive_service.files().list(
        q=query,
        spaces='drive',
        fields='files(id, name)',
        pageSize=10,
    ).execute()
    files = response.get('files', [])

    if len(files) > 1:
        raise RuntimeError(f'Duplicate spreadsheet name: {spreadsheet_name}')

    return files[0] if files else None


def move_file_to_folder(file_id, folder_id):
    file = drive_service.files().get(fileId=file_id, fields='parents').execute()
    previous_parents = ','.join(file.get('parents', []))
    drive_service.files().update(
        fileId=file_id,
        addParents=folder_id,
        removeParents=previous_parents,
        fields='id, parents',
    ).execute()


def get_or_create_worksheet(spreadsheet, title, rows=1, cols=1):
    try:
        return spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        return spreadsheet.add_worksheet(title=title, rows=rows, cols=cols)


data_folder = get_or_create_folder(FOLDER_ID, SPREADSHEET_FOLDER_PATH)
spreadsheet_file = find_spreadsheet_in_folder(data_folder['id'], SPREADSHEET_NAME)

if spreadsheet_file:
    spreadsheet = gc.open_by_key(spreadsheet_file['id'])
else:
    spreadsheet = gc.create(SPREADSHEET_NAME)
    move_file_to_folder(spreadsheet.id, data_folder['id'])

active_worksheet = get_or_create_worksheet(spreadsheet, ACTIVE_SHEET_NAME, rows=1, cols=len(df.columns))
get_or_create_worksheet(spreadsheet, STAGING_SHEET_NAME, rows=1, cols=len(df.columns))
metadata_worksheet = get_or_create_worksheet(spreadsheet, METADATA_SHEET_NAME, rows=1, cols=2)

active_worksheet.clear()
active_worksheet.resize(rows=len(df) + 1, cols=len(df.columns))

values = [df.columns.tolist()] + df.astype(object).values.tolist()
active_worksheet.update(values=values, range_name='A1', value_input_option='RAW')

metadata_values = [
    ['key', 'value'],
    ['last_refresh_started_at', ''],
    ['last_refresh_finished_at', datetime.now(timezone.utc).isoformat()],
    ['last_refresh_status', 'success'],
    ['last_refresh_trigger_type', 'colab'],
    ['last_refresh_error', ''],
    ['row_count', str(len(df))],
    ['schema_version', '1'],
    ['channel_id', ''],
    ['channel_title', ''],
    ['uploads_playlist_id', ''],
]
metadata_worksheet.clear()
metadata_worksheet.resize(rows=len(metadata_values), cols=2)
metadata_worksheet.update(values=metadata_values, range_name='A1', value_input_option='RAW')

print(f'path: MyDrive/ハヤトの野望/{SPREADSHEET_FOLDER_PATH}/{SPREADSHEET_NAME}')
print(f'spreadsheet: {spreadsheet.title}')
print(f'id: {spreadsheet.id}')
print(f'url: {spreadsheet.url}')
print(f'active sheet rows: {len(df)}')
print(f'columns: {len(df.columns)}')


In [ ]:
active_worksheet.get_all_values()[:3]
